# einops-reduce-min — worked example 2: Find the per-sequence minimum hidden state activation over timesteps

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-reduce-min`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When processing variable-length sequences with an RNN or Transformer, you sometimes need the minimum hidden state value across all timesteps for each sequence in the batch. The pattern `'b t d -> b d'` with `'min'` collapses the time axis, leaving one value per (batch, feature) pair. This is like a temporal global min-pool.

## Worked solution

Input: `(B=3, T=10, D=16)` hidden states from an LSTM — 3 sequences, 10 timesteps, 16-dimensional state.

**Pattern:** `'b t d -> b d'` with `'min'`.

For each sequence `b` and feature `d`, we scan all 10 timesteps and keep the minimum: `out[b, d] = min_t hidden[b, t, d]`.

**Result shape:** `(3, 16)`. Each sequence now has one 16-dim vector representing its minimum-activation signature across time.

In [ ]:
import torch as t
from einops import reduce

t.manual_seed(44)
B, T, D = 3, 12, 8
hidden = t.randn(B, T, D)

def temporal_min(hidden):
    """Min across timesteps: (B, T, D) -> (B, D)."""
    return reduce(hidden, 'b t d -> b d', 'min')

out = temporal_min(hidden)
print('Hidden shape:', hidden.shape)
print('Output shape:', out.shape)  # (3, 8)
assert out.shape == (B, D)

expected = hidden.min(dim=1).values
assert t.allclose(out, expected)
print('Matches hidden.min(dim=1).values:', True)

# Show: per-sequence, every feature value <= all timestep values
for b in range(B):
    assert (out[b] <= hidden[b].min(dim=0).values + 1e-6).all()
print('Per-sequence min invariant holds:', True)